In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../Data")

files = {
    "customers": "Sales Customer.csv",
    "orders": "Sales SalesOrderHeader.csv",
    "order_details": "Sales SalesOrderDetail.csv",
    "stores": "Sales Store.csv",
    "territories": "Sales SalesTerritory.csv",
    "products": "Production Product.csv",
    "subcategories": "Production ProductSubcategory.csv",
    "categories": "Production ProductCategory.csv"
}

tables = {}

for name, filename in files.items():
    tables[name] = pd.read_csv(data_path / filename)
    print(f"{name}: {tables[name].shape}")

customers: (19820, 7)
orders: (31465, 26)
order_details: (121317, 11)
stores: (701, 6)
territories: (10, 10)
products: (504, 25)
subcategories: (37, 5)
categories: (4, 4)


In [2]:
for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} — {df.shape}")
    print(df.columns.tolist())


CUSTOMERS — (19820, 7)
['CustomerID', 'PersonID', 'StoreID', 'TerritoryID', 'AccountNumber', 'rowguid', 'ModifiedDate']

ORDERS — (31465, 26)
['SalesOrderID', 'RevisionNumber', 'OrderDate', 'DueDate', 'ShipDate', 'Status', 'OnlineOrderFlag', 'SalesOrderNumber', 'PurchaseOrderNumber', 'AccountNumber', 'CustomerID', 'SalesPersonID', 'TerritoryID', 'BillToAddressID', 'ShipToAddressID', 'ShipMethodID', 'CreditCardID', 'CreditCardApprovalCode', 'CurrencyRateID', 'SubTotal', 'TaxAmt', 'Freight', 'TotalDue', 'Comment', 'rowguid', 'ModifiedDate']

ORDER_DETAILS — (121317, 11)
['SalesOrderID', 'SalesOrderDetailID', 'CarrierTrackingNumber', 'OrderQty', 'ProductID', 'SpecialOfferID', 'UnitPrice', 'UnitPriceDiscount', 'LineTotal', 'rowguid', 'ModifiedDate']

STORES — (701, 6)
['BusinessEntityID', 'Name', 'SalesPersonID', 'Demographics', 'rowguid', 'ModifiedDate']

TERRITORIES — (10, 10)
['TerritoryID', 'Name', 'CountryRegionCode', 'Group', 'SalesYTD', 'SalesLastYear', 'CostYTD', 'CostLastYear', '

In [3]:
for name, df in tables.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} DATA QUALITY")
    print(f"Rows: {len(df):,}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")
    print(f"Total missing values: {df.isna().sum().sum():,}")

    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("Columns with missing values:")
        print(missing)
    else:
        print("No missing values.")


CUSTOMERS DATA QUALITY
Rows: 19,820
Duplicate rows: 0
Total missing values: 19,185
Columns with missing values:
PersonID      701
StoreID     18484
dtype: int64

ORDERS DATA QUALITY
Rows: 31,465
Duplicate rows: 0
Total missing values: 106,534
Columns with missing values:
PurchaseOrderNumber       27659
SalesPersonID             27659
CreditCardID               1131
CreditCardApprovalCode     1131
CurrencyRateID            17489
Comment                   31465
dtype: int64

ORDER_DETAILS DATA QUALITY
Rows: 121,317
Duplicate rows: 0
Total missing values: 60,398
Columns with missing values:
CarrierTrackingNumber    60398
dtype: int64

STORES DATA QUALITY
Rows: 701
Duplicate rows: 0
Total missing values: 0
No missing values.

TERRITORIES DATA QUALITY
Rows: 10
Duplicate rows: 0
Total missing values: 0
No missing values.

PRODUCTS DATA QUALITY
Rows: 504
Duplicate rows: 0
Total missing values: 3,571
Columns with missing values:
Color                    248
Size                     293
SizeUn

In [4]:
# Referential integrity checks

checks = {
    "Orders → Customers": (
        tables["orders"]["CustomerID"],
        tables["customers"]["CustomerID"]
    ),
    "Orders → Territories": (
        tables["orders"]["TerritoryID"],
        tables["territories"]["TerritoryID"]
    ),
    "Order Details → Orders": (
        tables["order_details"]["SalesOrderID"],
        tables["orders"]["SalesOrderID"]
    ),
    "Order Details → Products": (
        tables["order_details"]["ProductID"],
        tables["products"]["ProductID"]
    ),
    "Products → Subcategories": (
        tables["products"]["ProductSubcategoryID"],
        tables["subcategories"]["ProductSubcategoryID"]
    ),
    "Subcategories → Categories": (
        tables["subcategories"]["ProductCategoryID"],
        tables["categories"]["ProductCategoryID"]
    ),
    "Customers → Stores": (
        tables["customers"]["StoreID"],
        tables["stores"]["BusinessEntityID"]
    )
}

for relationship, (child, parent) in checks.items():
    unmatched = child.dropna()[~child.dropna().isin(parent)].nunique()

    print(f"{relationship}: {unmatched} unmatched key(s)")

Orders → Customers: 0 unmatched key(s)
Orders → Territories: 0 unmatched key(s)
Order Details → Orders: 0 unmatched key(s)
Order Details → Products: 0 unmatched key(s)
Products → Subcategories: 0 unmatched key(s)
Subcategories → Categories: 0 unmatched key(s)
Customers → Stores: 0 unmatched key(s)


In [5]:
primary_keys = {
    "Customers": ("CustomerID", tables["customers"]),
    "Orders": ("SalesOrderID", tables["orders"]),
    "Order Details": ("SalesOrderDetailID", tables["order_details"]),
    "Stores": ("BusinessEntityID", tables["stores"]),
    "Territories": ("TerritoryID", tables["territories"]),
    "Products": ("ProductID", tables["products"]),
    "Subcategories": ("ProductSubcategoryID", tables["subcategories"]),
    "Categories": ("ProductCategoryID", tables["categories"])
}

for name, (key, df) in primary_keys.items():
    duplicates = df[key].duplicated().sum()
    nulls = df[key].isna().sum()

    print(
        f"{name}: "
        f"{key} | duplicates = {duplicates:,} | nulls = {nulls:,}"
    )

Customers: CustomerID | duplicates = 0 | nulls = 0
Orders: SalesOrderID | duplicates = 0 | nulls = 0
Order Details: SalesOrderDetailID | duplicates = 0 | nulls = 0
Stores: BusinessEntityID | duplicates = 0 | nulls = 0
Territories: TerritoryID | duplicates = 0 | nulls = 0
Products: ProductID | duplicates = 0 | nulls = 0
Subcategories: ProductSubcategoryID | duplicates = 0 | nulls = 0
Categories: ProductCategoryID | duplicates = 0 | nulls = 0


In [6]:
# Create clean copies for ETL processing

customers = tables["customers"].copy()
orders = tables["orders"].copy()
order_details = tables["order_details"].copy()
stores = tables["stores"].copy()
territories = tables["territories"].copy()
products = tables["products"].copy()
subcategories = tables["subcategories"].copy()
categories = tables["categories"].copy()

print("Clean staging tables created:")
for name, df in {
    "customers": customers,
    "orders": orders,
    "order_details": order_details,
    "stores": stores,
    "territories": territories,
    "products": products,
    "subcategories": subcategories,
    "categories": categories
}.items():
    print(f"{name}: {df.shape}")

Clean staging tables created:
customers: (19820, 7)
orders: (31465, 26)
order_details: (121317, 11)
stores: (701, 6)
territories: (10, 10)
products: (504, 25)
subcategories: (37, 5)
categories: (4, 4)


In [7]:
# Convert date columns to proper datetime format

orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], errors="coerce")
orders["DueDate"] = pd.to_datetime(orders["DueDate"], errors="coerce")
orders["ShipDate"] = pd.to_datetime(orders["ShipDate"], errors="coerce")

products["SellStartDate"] = pd.to_datetime(
    products["SellStartDate"], errors="coerce"
)
products["SellEndDate"] = pd.to_datetime(
    products["SellEndDate"], errors="coerce"
)
products["DiscontinuedDate"] = pd.to_datetime(
    products["DiscontinuedDate"], errors="coerce"
)

print("Orders:")
print(orders[["OrderDate", "DueDate", "ShipDate"]].dtypes)

print("\nProducts:")
print(products[["SellStartDate", "SellEndDate", "DiscontinuedDate"]].dtypes)

Orders:
OrderDate    datetime64[us]
DueDate      datetime64[us]
ShipDate     datetime64[us]
dtype: object

Products:
SellStartDate       datetime64[us]
SellEndDate         datetime64[us]
DiscontinuedDate     datetime64[s]
dtype: object


In [8]:
# Validate order date consistency

print("Missing dates:")
print(orders[["OrderDate", "DueDate", "ShipDate"]].isna().sum())

print("\nDate inconsistencies:")

order_after_ship = (orders["OrderDate"] > orders["ShipDate"]).sum()
due_before_order = (orders["DueDate"] < orders["OrderDate"]).sum()
ship_after_due = (orders["ShipDate"] > orders["DueDate"]).sum()

print(f"Order Date > Ship Date: {order_after_ship:,}")
print(f"Due Date < Order Date: {due_before_order:,}")
print(f"Ship Date > Due Date: {ship_after_due:,}")

Missing dates:
OrderDate    0
DueDate      0
ShipDate     0
dtype: int64

Date inconsistencies:
Order Date > Ship Date: 0
Due Date < Order Date: 0
Ship Date > Due Date: 0


In [9]:
# Validate order-detail financial calculations

order_details["CalculatedLineTotal"] = (
    order_details["OrderQty"] *
    order_details["UnitPrice"] *
    (1 - order_details["UnitPriceDiscount"])
)

order_details["Difference"] = (
    order_details["LineTotal"] -
    order_details["CalculatedLineTotal"]
)

print("Negative quantities:", (order_details["OrderQty"] < 0).sum())
print("Negative unit prices:", (order_details["UnitPrice"] < 0).sum())
print("Negative line totals:", (order_details["LineTotal"] < 0).sum())
print("Negative discounts:", (order_details["UnitPriceDiscount"] < 0).sum())
print("Discounts above 100%:", (order_details["UnitPriceDiscount"] > 1).sum())

print("\nLine-total differences:")
print("Maximum absolute difference:",
      order_details["Difference"].abs().max())

print("Rows with difference > 0.01:",
      (order_details["Difference"].abs() > 0.01).sum())

Negative quantities: 0
Negative unit prices: 0
Negative line totals: 0
Negative discounts: 0
Discounts above 100%: 0

Line-total differences:
Maximum absolute difference: 3.637978807091713e-12
Rows with difference > 0.01: 0


In [10]:
# Validate order-level subtotals against order-detail totals

detail_totals = (
    order_details
    .groupby("SalesOrderID")["LineTotal"]
    .sum()
    .rename("CalculatedSubTotal")
)

order_validation = orders[
    ["SalesOrderID", "SubTotal", "TaxAmt", "Freight", "TotalDue"]
].merge(
    detail_totals,
    on="SalesOrderID",
    how="left"
)

order_validation["SubTotalDifference"] = (
    order_validation["SubTotal"]
    - order_validation["CalculatedSubTotal"]
)

print("Orders without detail records:",
      order_validation["CalculatedSubTotal"].isna().sum())

print(
    "Orders with SubTotal difference > $0.01:",
    (order_validation["SubTotalDifference"].abs() > 0.01).sum()
)

print(
    "Maximum SubTotal difference: $",
    round(order_validation["SubTotalDifference"].abs().max(), 6)
)

Orders without detail records: 0
Orders with SubTotal difference > $0.01: 0
Maximum SubTotal difference: $ 5e-05


In [11]:
# Create Product Dimension

dim_product = (
    products[
        [
            "ProductID",
            "Name",
            "ProductNumber",
            "Color",
            "Size",
            "StandardCost",
            "ListPrice",
            "ProductLine",
            "Class",
            "Style",
            "ProductSubcategoryID"
        ]
    ]
    .merge(
        subcategories[
            ["ProductSubcategoryID", "ProductCategoryID"]
        ],
        on="ProductSubcategoryID",
        how="left"
    )
    .merge(
        categories[
            ["ProductCategoryID", "Name"]
        ],
        on="ProductCategoryID",
        how="left",
        suffixes=("", "_Category")
    )
)

dim_product = dim_product.rename(
    columns={
        "ProductID": "ProductKey",
        "Name": "ProductName",
        "Name_Category": "CategoryName"
    }
)

print("dim_product shape:", dim_product.shape)
print("\nColumns:")
print(dim_product.columns.tolist())

dim_product.head()

dim_product shape: (504, 13)

Columns:
['ProductKey', 'ProductName', 'ProductNumber', 'Color', 'Size', 'StandardCost', 'ListPrice', 'ProductLine', 'Class', 'Style', 'ProductSubcategoryID', 'ProductCategoryID', 'CategoryName']


,ProductKey,ProductName,ProductNumber,Color,Size,StandardCost,ListPrice,ProductLine,Class,Style,ProductSubcategoryID,ProductCategoryID,CategoryName
0,1,Adjustable Race,AR-5381,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Bearing Ball,BA-8327,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,323,Crown Race,CR-9981,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,325,Decal 1,DC-8732,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,326,Decal 2,DC-9824,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Handle missing product hierarchy values

dim_product["ProductSubcategoryID"] = (
    dim_product["ProductSubcategoryID"].fillna(-1)
)

dim_product["ProductCategoryID"] = (
    dim_product["ProductCategoryID"].fillna(-1)
)

dim_product["CategoryName"] = (
    dim_product["CategoryName"].fillna("Unknown")
)

# Validate
print("Total products:", len(dim_product))
print(
    "Products with Unknown category:",
    (dim_product["CategoryName"] == "Unknown").sum()
)

print("\nCategory distribution:")
print(dim_product["CategoryName"].value_counts())

Total products: 504
Products with Unknown category: 209

Category distribution:
CategoryName
Unknown        209
Components     134
Bikes           97
Clothing        35
Accessories     29
Name: count, dtype: int64


In [13]:
# Validate Product Dimension

print("Rows:", len(dim_product))
print("Duplicate ProductKeys:", dim_product["ProductKey"].duplicated().sum())
print("Null ProductKeys:", dim_product["ProductKey"].isna().sum())

print("\nNull values by column:")
print(dim_product.isna().sum())

Rows: 504
Duplicate ProductKeys: 0
Null ProductKeys: 0

Null values by column:
ProductKey                0
ProductName               0
ProductNumber             0
Color                   248
Size                    293
StandardCost              0
ListPrice                 0
ProductLine             226
Class                   257
Style                   293
ProductSubcategoryID      0
ProductCategoryID         0
CategoryName              0
dtype: int64


In [14]:
# Create Customer Dimension

dim_customer = customers[
    [
        "CustomerID",
        "PersonID",
        "StoreID",
        "TerritoryID",
        "AccountNumber"
    ]
].copy()

dim_customer = dim_customer.rename(
    columns={
        "CustomerID": "CustomerKey",
        "PersonID": "PersonKey",
        "StoreID": "StoreKey",
        "TerritoryID": "TerritoryKey",
        "AccountNumber": "AccountNumber"
    }
)

print("dim_customer shape:", dim_customer.shape)
print("\nColumns:")
print(dim_customer.columns.tolist())

dim_customer.head()

dim_customer shape: (19820, 5)

Columns:
['CustomerKey', 'PersonKey', 'StoreKey', 'TerritoryKey', 'AccountNumber']


,CustomerKey,PersonKey,StoreKey,TerritoryKey,AccountNumber
0,11015,10963.0,NaN,4,AW00011015
1,11016,3800.0,NaN,4,AW00011016
2,11023,4373.0,NaN,4,AW00011023
3,11024,16843.0,NaN,4,AW00011024
4,11036,20539.0,NaN,4,AW00011036


In [15]:
# Validate Customer Dimension

print("Rows:", len(dim_customer))
print("Duplicate CustomerKeys:",
      dim_customer["CustomerKey"].duplicated().sum())
print("Null CustomerKeys:",
      dim_customer["CustomerKey"].isna().sum())

print("\nNull values by column:")
print(dim_customer.isna().sum())

Rows: 19820
Duplicate CustomerKeys: 0
Null CustomerKeys: 0

Null values by column:
CustomerKey          0
PersonKey          701
StoreKey         18484
TerritoryKey         0
AccountNumber        0
dtype: int64


In [16]:
# Create Store Dimension

dim_store = stores[
    [
        "BusinessEntityID",
        "Name",
        "SalesPersonID"
    ]
].copy()

dim_store = dim_store.rename(
    columns={
        "BusinessEntityID": "StoreKey",
        "Name": "StoreName",
        "SalesPersonID": "SalesPersonKey"
    }
)

print("dim_store shape:", dim_store.shape)
print("\nColumns:")
print(dim_store.columns.tolist())

dim_store.head()

dim_store shape: (701, 3)

Columns:
['StoreKey', 'StoreName', 'SalesPersonKey']


,StoreKey,StoreName,SalesPersonKey
0,292,Next-Door Bike Store,279
1,310,New Bikes Company,279
2,326,Worthwhile Activity Store,279
3,334,Global Plaza,279
4,340,eCommerce Bikes,279


In [17]:
# Validate Store Dimension

print("Rows:", len(dim_store))
print("Duplicate StoreKeys:",
      dim_store["StoreKey"].duplicated().sum())
print("Null StoreKeys:",
      dim_store["StoreKey"].isna().sum())

print("\nNull values by column:")
print(dim_store.isna().sum())

Rows: 701
Duplicate StoreKeys: 0
Null StoreKeys: 0

Null values by column:
StoreKey          0
StoreName         0
SalesPersonKey    0
dtype: int64


In [18]:
# Create Territory Dimension

dim_territory = territories[
    [
        "TerritoryID",
        "Name",
        "CountryRegionCode",
        "Group"
    ]
].copy()

dim_territory = dim_territory.rename(
    columns={
        "TerritoryID": "TerritoryKey",
        "Name": "TerritoryName",
        "CountryRegionCode": "CountryCode",
        "Group": "RegionGroup"
    }
)

print("dim_territory shape:", dim_territory.shape)
print("\nColumns:")
print(dim_territory.columns.tolist())

dim_territory.head()

dim_territory shape: (10, 4)

Columns:
['TerritoryKey', 'TerritoryName', 'CountryCode', 'RegionGroup']


,TerritoryKey,TerritoryName,CountryCode,RegionGroup
0,1,Northwest,US,North America
1,2,Northeast,US,North America
2,3,Central,US,North America
3,4,Southwest,US,North America
4,5,Southeast,US,North America


In [19]:
# Validate Territory Dimension

print("Rows:", len(dim_territory))
print("Duplicate TerritoryKeys:",
      dim_territory["TerritoryKey"].duplicated().sum())
print("Null TerritoryKeys:",
      dim_territory["TerritoryKey"].isna().sum())

print("\nNull values by column:")
print(dim_territory.isna().sum())

Rows: 10
Duplicate TerritoryKeys: 0
Null TerritoryKeys: 0

Null values by column:
TerritoryKey     0
TerritoryName    0
CountryCode      0
RegionGroup      0
dtype: int64


In [20]:
# Create Date Dimension

date_range = pd.date_range(
    start=orders["OrderDate"].min(),
    end=orders["OrderDate"].max(),
    freq="D"
)

dim_date = pd.DataFrame({
    "Date": date_range
})

dim_date["DateKey"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = "Q" + dim_date["Date"].dt.quarter.astype(str)
dim_date["MonthNumber"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.strftime("%B")
dim_date["MonthShort"] = dim_date["Date"].dt.strftime("%b")
dim_date["WeekNumber"] = dim_date["Date"].dt.isocalendar().week.astype(int)
dim_date["DayOfMonth"] = dim_date["Date"].dt.day
dim_date["DayName"] = dim_date["Date"].dt.strftime("%A")
dim_date["DayOfWeek"] = dim_date["Date"].dt.dayofweek + 1

print("dim_date shape:", dim_date.shape)
print("\nDate range:")
print(dim_date["Date"].min(), "to", dim_date["Date"].max())

print("\nColumns:")
print(dim_date.columns.tolist())

dim_date.head()

dim_date shape: (1127, 11)

Date range:
2011-05-31 00:00:00 to 2014-06-30 00:00:00

Columns:
['Date', 'DateKey', 'Year', 'Quarter', 'MonthNumber', 'MonthName', 'MonthShort', 'WeekNumber', 'DayOfMonth', 'DayName', 'DayOfWeek']


,Date,DateKey,Year,Quarter,MonthNumber,MonthName,MonthShort,WeekNumber,DayOfMonth,DayName,DayOfWeek
0,2011-05-31,20110531,2011,Q2,5,May,May,22,31,Tuesday,2
1,2011-06-01,20110601,2011,Q2,6,June,Jun,22,1,Wednesday,3
2,2011-06-02,20110602,2011,Q2,6,June,Jun,22,2,Thursday,4
3,2011-06-03,20110603,2011,Q2,6,June,Jun,22,3,Friday,5
4,2011-06-04,20110604,2011,Q2,6,June,Jun,22,4,Saturday,6


In [21]:
# Validate Date Dimension

print("Rows:", len(dim_date))
print("Duplicate DateKeys:",
      dim_date["DateKey"].duplicated().sum())
print("Null DateKeys:",
      dim_date["DateKey"].isna().sum())
print("Duplicate Dates:",
      dim_date["Date"].duplicated().sum())
print("Null Dates:",
      dim_date["Date"].isna().sum())

print("\nDate range:")
print(dim_date["Date"].min(), "to", dim_date["Date"].max())

Rows: 1127
Duplicate DateKeys: 0
Null DateKeys: 0
Duplicate Dates: 0
Null Dates: 0

Date range:
2011-05-31 00:00:00 to 2014-06-30 00:00:00


In [22]:
# Create Sales Fact Table

fact_sales = order_details[
    [
        "SalesOrderID",
        "SalesOrderDetailID",
        "ProductID",
        "OrderQty",
        "UnitPrice",
        "UnitPriceDiscount",
        "LineTotal"
    ]
].copy()

# Add order-level information
fact_sales = fact_sales.merge(
    orders[
        [
            "SalesOrderID",
            "OrderDate",
            "CustomerID",
            "TerritoryID"
        ]
    ],
    on="SalesOrderID",
    how="left"
)

# Add warehouse keys
fact_sales["DateKey"] = (
    fact_sales["OrderDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

fact_sales = fact_sales.rename(
    columns={
        "CustomerID": "CustomerKey",
        "ProductID": "ProductKey",
        "TerritoryID": "TerritoryKey",
        "SalesOrderID": "OrderKey",
        "SalesOrderDetailID": "SalesOrderDetailKey",
        "OrderQty": "Quantity",
        "UnitPrice": "UnitPrice",
        "UnitPriceDiscount": "DiscountRate",
        "LineTotal": "SalesAmount"
    }
)

# Select final fact table columns
fact_sales = fact_sales[
    [
        "SalesOrderDetailKey",
        "OrderKey",
        "DateKey",
        "CustomerKey",
        "ProductKey",
        "TerritoryKey",
        "Quantity",
        "UnitPrice",
        "DiscountRate",
        "SalesAmount"
    ]
]

print("fact_sales shape:", fact_sales.shape)

print("\nColumns:")
print(fact_sales.columns.tolist())

print("\nSample:")
fact_sales.head()

fact_sales shape: (121317, 10)

Columns:
['SalesOrderDetailKey', 'OrderKey', 'DateKey', 'CustomerKey', 'ProductKey', 'TerritoryKey', 'Quantity', 'UnitPrice', 'DiscountRate', 'SalesAmount']

Sample:


,SalesOrderDetailKey,OrderKey,DateKey,CustomerKey,ProductKey,TerritoryKey,Quantity,UnitPrice,DiscountRate,SalesAmount
0,37753,51178,20130530,11245,870,8,1,4.99,0.0,4.99
1,37760,51180,20130530,16313,870,8,1,4.99,0.0,4.99
2,37790,51191,20130531,12390,870,8,1,4.99,0.0,4.99
3,37804,51196,20130531,18906,870,9,1,4.99,0.0,4.99
4,37809,51197,20130531,11448,870,9,1,4.99,0.0,4.99


In [23]:
# Validate Sales Fact Table

print("Rows:", len(fact_sales))

print("\nKey validation:")
for key in [
    "SalesOrderDetailKey",
    "OrderKey",
    "DateKey",
    "CustomerKey",
    "ProductKey",
    "TerritoryKey"
]:
    print(
        f"{key}: "
        f"duplicates={fact_sales[key].duplicated().sum():,}, "
        f"nulls={fact_sales[key].isna().sum():,}"
    )

print("\nFinancial validation:")
print("Negative Quantity:",
      (fact_sales["Quantity"] < 0).sum())
print("Negative UnitPrice:",
      (fact_sales["UnitPrice"] < 0).sum())
print("Negative SalesAmount:",
      (fact_sales["SalesAmount"] < 0).sum())
print("DiscountRate > 100%:",
      (fact_sales["DiscountRate"] > 1).sum())

print("\nSales totals:")
print("Total Sales Amount: $",
      round(fact_sales["SalesAmount"].sum(), 2))

print("Total Quantity:",
      fact_sales["Quantity"].sum())

Rows: 121317

Key validation:
SalesOrderDetailKey: duplicates=0, nulls=0
OrderKey: duplicates=89,852, nulls=0
DateKey: duplicates=120,193, nulls=0
CustomerKey: duplicates=102,198, nulls=0
ProductKey: duplicates=121,051, nulls=0
TerritoryKey: duplicates=121,307, nulls=0

Financial validation:
Negative Quantity: 0
Negative UnitPrice: 0
Negative SalesAmount: 0
DiscountRate > 100%: 0

Sales totals:
Total Sales Amount: $ 109846381.4
Total Quantity: 274914


In [24]:
# Export transformed warehouse tables

output_path = Path("../ETL")

warehouse_tables = {
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_store": dim_store,
    "dim_territory": dim_territory,
    "dim_date": dim_date,
    "fact_sales": fact_sales
}

for name, df in warehouse_tables.items():
    file_path = output_path / f"{name}.csv"
    df.to_csv(file_path, index=False)
    print(f"Saved: {file_path} | Rows: {len(df):,}")

Saved: ..\ETL\dim_product.csv | Rows: 504
Saved: ..\ETL\dim_customer.csv | Rows: 19,820
Saved: ..\ETL\dim_store.csv | Rows: 701
Saved: ..\ETL\dim_territory.csv | Rows: 10
Saved: ..\ETL\dim_date.csv | Rows: 1,127
Saved: ..\ETL\fact_sales.csv | Rows: 121,317


In [25]:
# Reorder dim_date columns to match PostgreSQL

dim_date = dim_date[
    [
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "MonthShort",
        "WeekNumber",
        "DayOfMonth",
        "DayName",
        "DayOfWeek"
    ]
]

dim_date.to_csv(
    "../ETL/dim_date.csv",
    index=False
)

print("dim_date.csv recreated successfully.")
print(dim_date.columns.tolist())

dim_date.csv recreated successfully.
['DateKey', 'Date', 'Year', 'Quarter', 'MonthNumber', 'MonthName', 'MonthShort', 'WeekNumber', 'DayOfMonth', 'DayName', 'DayOfWeek']


In [26]:
# Reorder and export all remaining warehouse tables
# to match the PostgreSQL table definitions.

dim_customer = dim_customer[
    [
        "CustomerKey",
        "PersonKey",
        "StoreKey",
        "TerritoryKey",
        "AccountNumber"
    ]
]

dim_store = dim_store[
    [
        "StoreKey",
        "StoreName",
        "SalesPersonKey"
    ]
]

dim_territory = dim_territory[
    [
        "TerritoryKey",
        "TerritoryName",
        "CountryCode",
        "RegionGroup"
    ]
]

dim_product = dim_product[
    [
        "ProductKey",
        "ProductName",
        "ProductNumber",
        "Color",
        "Size",
        "StandardCost",
        "ListPrice",
        "ProductLine",
        "Class",
        "Style",
        "ProductSubcategoryID",
        "ProductCategoryID",
        "CategoryName"
    ]
].rename(
    columns={
        "ProductSubcategoryID": "ProductSubcategoryKey",
        "ProductCategoryID": "ProductCategoryKey"
    }
)

fact_sales = fact_sales[
    [
        "SalesOrderDetailKey",
        "OrderKey",
        "DateKey",
        "CustomerKey",
        "ProductKey",
        "TerritoryKey",
        "Quantity",
        "UnitPrice",
        "DiscountRate",
        "SalesAmount"
    ]
]

exports = {
    "dim_customer": dim_customer,
    "dim_store": dim_store,
    "dim_territory": dim_territory,
    "dim_product": dim_product,
    "fact_sales": fact_sales
}

for name, df in exports.items():
    df.to_csv(f"../ETL/{name}.csv", index=False)
    print(f"{name}.csv → {df.shape}")

dim_customer.csv → (19820, 5)
dim_store.csv → (701, 3)
dim_territory.csv → (10, 4)
dim_product.csv → (504, 13)
fact_sales.csv → (121317, 10)


In [27]:
# Fix nullable integer keys before exporting to PostgreSQL

dim_customer["CustomerKey"] = dim_customer["CustomerKey"].astype("int64")

dim_customer["PersonKey"] = dim_customer["PersonKey"].astype("Int64")
dim_customer["StoreKey"] = dim_customer["StoreKey"].astype("Int64")
dim_customer["TerritoryKey"] = dim_customer["TerritoryKey"].astype("int64")

dim_customer.to_csv(
    "../ETL/dim_customer.csv",
    index=False
)

print(dim_customer.dtypes)
print("\nSample:")
print(dim_customer.head())

CustomerKey      int64
PersonKey        Int64
StoreKey         Int64
TerritoryKey     int64
AccountNumber      str
dtype: object

Sample:
   CustomerKey  PersonKey  StoreKey  TerritoryKey AccountNumber
0        11015      10963      <NA>             4    AW00011015
1        11016       3800      <NA>             4    AW00011016
2        11023       4373      <NA>             4    AW00011023
3        11024      16843      <NA>             4    AW00011024
4        11036      20539      <NA>             4    AW00011036


In [28]:
# Prepare Store Dimension for PostgreSQL

dim_store["StoreKey"] = dim_store["StoreKey"].astype("int64")
dim_store["SalesPersonKey"] = dim_store["SalesPersonKey"].astype("Int64")

dim_store.to_csv(
    "../ETL/dim_store.csv",
    index=False
)

print(dim_store.dtypes)
print("\nMissing values:")
print(dim_store.isna().sum())

StoreKey          int64
StoreName           str
SalesPersonKey    Int64
dtype: object

Missing values:
StoreKey          0
StoreName         0
SalesPersonKey    0
dtype: int64


In [29]:
# Prepare Territory Dimension for PostgreSQL

dim_territory["TerritoryKey"] = dim_territory["TerritoryKey"].astype("int64")
dim_territory["TerritoryName"] = dim_territory["TerritoryName"].astype(str)
dim_territory["CountryCode"] = dim_territory["CountryCode"].astype(str)
dim_territory["RegionGroup"] = dim_territory["RegionGroup"].astype(str)

dim_territory.to_csv(
    "../ETL/dim_territory.csv",
    index=False
)

print(dim_territory.dtypes)
print("\nMissing values:")
print(dim_territory.isna().sum())

TerritoryKey     int64
TerritoryName      str
CountryCode        str
RegionGroup        str
dtype: object

Missing values:
TerritoryKey     0
TerritoryName    0
CountryCode      0
RegionGroup      0
dtype: int64


In [30]:
# Prepare Product Dimension for PostgreSQL

dim_product["ProductKey"] = dim_product["ProductKey"].astype("int64")
dim_product["ProductSubcategoryKey"] = (
    dim_product["ProductSubcategoryKey"].astype("int64")
)
dim_product["ProductCategoryKey"] = (
    dim_product["ProductCategoryKey"].astype("int64")
)

dim_product["StandardCost"] = dim_product["StandardCost"].astype(float)
dim_product["ListPrice"] = dim_product["ListPrice"].astype(float)

dim_product.to_csv(
    "../ETL/dim_product.csv",
    index=False
)

print(dim_product.dtypes)

print("\nShape:", dim_product.shape)

print("\nMissing values:")
print(dim_product.isna().sum())

ProductKey                 int64
ProductName                  str
ProductNumber                str
Color                        str
Size                         str
StandardCost             float64
ListPrice                float64
ProductLine                  str
Class                        str
Style                        str
ProductSubcategoryKey      int64
ProductCategoryKey         int64
CategoryName                 str
dtype: object

Shape: (504, 13)

Missing values:
ProductKey                 0
ProductName                0
ProductNumber              0
Color                    248
Size                     293
StandardCost               0
ListPrice                  0
ProductLine              226
Class                    257
Style                    293
ProductSubcategoryKey      0
ProductCategoryKey         0
CategoryName               0
dtype: int64


In [31]:
# Prepare Fact Sales for PostgreSQL

integer_columns = [
    "SalesOrderDetailKey",
    "OrderKey",
    "DateKey",
    "CustomerKey",
    "ProductKey",
    "TerritoryKey",
    "Quantity"
]

for col in integer_columns:
    fact_sales[col] = fact_sales[col].astype("int64")

fact_sales["UnitPrice"] = fact_sales["UnitPrice"].astype(float)
fact_sales["DiscountRate"] = fact_sales["DiscountRate"].astype(float)
fact_sales["SalesAmount"] = fact_sales["SalesAmount"].astype(float)

fact_sales.to_csv(
    "../ETL/fact_sales.csv",
    index=False
)

print("Shape:", fact_sales.shape)
print("\nData types:")
print(fact_sales.dtypes)

print("\nMissing values:")
print(fact_sales.isna().sum())

Shape: (121317, 10)

Data types:
SalesOrderDetailKey      int64
OrderKey                 int64
DateKey                  int64
CustomerKey              int64
ProductKey               int64
TerritoryKey             int64
Quantity                 int64
UnitPrice              float64
DiscountRate           float64
SalesAmount            float64
dtype: object

Missing values:
SalesOrderDetailKey    0
OrderKey               0
DateKey                0
CustomerKey            0
ProductKey             0
TerritoryKey           0
Quantity               0
UnitPrice              0
DiscountRate           0
SalesAmount            0
dtype: int64


In [32]:
# Check LineTotal decimal precision in the source data

line_total_4dp = (
    order_details["LineTotal"]
    .round(4)
)

rows_with_more_than_2_decimals = (
    (line_total_4dp * 100) % 1 != 0
).sum()

print(
    "Rows with more than 2 decimal places:",
    rows_with_more_than_2_decimals
)

print(
    "Original LineTotal sum:",
    order_details["LineTotal"].sum()
)

Rows with more than 2 decimal places: 58165
Original LineTotal sum: 109846381.39988798


In [33]:
# Check whether 4 decimal places are enough

line_total_6dp = order_details["LineTotal"].round(6)

rows_more_than_4_decimals = (
    (line_total_6dp * 10000) % 1 != 0
).sum()

print(
    "Rows with more than 4 decimal places:",
    rows_more_than_4_decimals
)

Rows with more than 4 decimal places: 15100


In [34]:
# Check whether 6 decimal places preserve all source precision

line_total_8dp = order_details["LineTotal"].round(8)

rows_more_than_6_decimals = (
    (line_total_8dp * 1_000_000) % 1 != 0
).sum()

print(
    "Rows with more than 6 decimal places:",
    rows_more_than_6_decimals
)


Rows with more than 6 decimal places: 1153


In [35]:
# Remove floating-point noise while preserving useful precision

original_total = order_details["LineTotal"].sum()

rounded_total = order_details["LineTotal"].round(6).sum()

difference = original_total - rounded_total

print("Original total:", original_total)
print("6-decimal total:", rounded_total)
print("Difference:", difference)

Original total: 109846381.39988798
6-decimal total: 109846381.39988798
Difference: 0.0


In [36]:
# Preserve financial precision for PostgreSQL

fact_sales["UnitPrice"] = fact_sales["UnitPrice"].round(6)
fact_sales["DiscountRate"] = fact_sales["DiscountRate"].round(6)
fact_sales["SalesAmount"] = fact_sales["SalesAmount"].round(6)

fact_sales.to_csv(
    "../ETL/fact_sales.csv",
    index=False,
    float_format="%.6f"
)

print("fact_sales.csv recreated.")
print("Rows:", len(fact_sales))
print("Sales total:", fact_sales["SalesAmount"].sum())

fact_sales.csv recreated.
Rows: 121317
Sales total: 109846381.39988798


In [37]:
print(fact_sales.shape)

(121317, 10)


In [38]:
print(fact_sales.shape)

(121317, 10)


In [39]:
# Add StoreKey to the sales fact table

customer_store = customers[
    ["CustomerID", "StoreID"]
].copy()

customer_store = customer_store.rename(
    columns={
        "CustomerID": "CustomerKey",
        "StoreID": "StoreKey"
    }
)

fact_sales = fact_sales.merge(
    customer_store,
    on="CustomerKey",
    how="left"
)

fact_sales = fact_sales[
    [
        "SalesOrderDetailKey",
        "OrderKey",
        "DateKey",
        "CustomerKey",
        "StoreKey",
        "ProductKey",
        "TerritoryKey",
        "Quantity",
        "UnitPrice",
        "DiscountRate",
        "SalesAmount"
    ]
]

print("Shape:", fact_sales.shape)
print("\nColumns:")
print(fact_sales.columns.tolist())
print("\nMissing StoreKey:", fact_sales["StoreKey"].isna().sum())

Shape: (121317, 11)

Columns:
['SalesOrderDetailKey', 'OrderKey', 'DateKey', 'CustomerKey', 'StoreKey', 'ProductKey', 'TerritoryKey', 'Quantity', 'UnitPrice', 'DiscountRate', 'SalesAmount']

Missing StoreKey: 60398


In [40]:
# Prepare updated fact_sales with StoreKey

fact_sales["StoreKey"] = fact_sales["StoreKey"].astype("Int64")

fact_sales["UnitPrice"] = fact_sales["UnitPrice"].round(6)
fact_sales["DiscountRate"] = fact_sales["DiscountRate"].round(6)
fact_sales["SalesAmount"] = fact_sales["SalesAmount"].round(6)

fact_sales.to_csv(
    "../ETL/fact_sales.csv",
    index=False,
    float_format="%.6f"
)

print("Shape:", fact_sales.shape)
print("\nColumns:")
print(fact_sales.columns.tolist())
print("\nMissing StoreKey:", fact_sales["StoreKey"].isna().sum())
print("\nSales total:", fact_sales["SalesAmount"].sum())

Shape: (121317, 11)

Columns:
['SalesOrderDetailKey', 'OrderKey', 'DateKey', 'CustomerKey', 'StoreKey', 'ProductKey', 'TerritoryKey', 'Quantity', 'UnitPrice', 'DiscountRate', 'SalesAmount']

Missing StoreKey: 60398

Sales total: 109846381.39988798


In [41]:
# Match CSV column order to PostgreSQL fact_sales table

fact_sales = fact_sales[
    [
        "SalesOrderDetailKey",
        "OrderKey",
        "DateKey",
        "CustomerKey",
        "ProductKey",
        "TerritoryKey",
        "Quantity",
        "UnitPrice",
        "DiscountRate",
        "SalesAmount",
        "StoreKey"
    ]
]

fact_sales.to_csv(
    "../ETL/fact_sales.csv",
    index=False,
    float_format="%.6f"
)

print(fact_sales.columns.tolist())
print("Rows:", len(fact_sales))
print("Sales total:", fact_sales["SalesAmount"].sum())

['SalesOrderDetailKey', 'OrderKey', 'DateKey', 'CustomerKey', 'ProductKey', 'TerritoryKey', 'Quantity', 'UnitPrice', 'DiscountRate', 'SalesAmount', 'StoreKey']
Rows: 121317
Sales total: 109846381.39988798
